# FIX - Emotion Notebook

FIX is built using the `exlib` library, which we load using a local version for now. You can uncomment the `!pip install exlib` line and comment out the `import sys; sys.path.insert(0, "../../src")` line if you do not have a local version you are testing.

In [ ]:
# Uncomment line below to install exlib
!pip install exlib
!pip install captum
!pip install numpy
import sys; sys.path.insert(0, "../../src")
import exlib

  Using cached numpy-2.3.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
captum 0.8.0 requires numpy<2.0, but you have numpy 2.0.2 which is incompatible.


  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
^C


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import tqdm
from tqdm import tqdm
from torch.utils.data import DataLoader
from datasets import load_dataset
import torch.nn as nn
import sentence_transformers

from exlib.datasets.emotion_helper import project_points_onto_axes, load_emotions
from exlib.datasets.emotion import load_data, load_model, EmotionDataset, EmotionClassifier, EmotionFixScore, get_emotion_scores

from exlib.features.text import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Load datasets and pre-trained models

In [ ]:
dataset = EmotionDataset("test")
dataloader = DataLoader(dataset, batch_size=2, shuffle=False)
model = EmotionClassifier().eval().to(device)

SamLowe/roberta-base-go_emotions


### Model prediction

In [ ]:
for batch in tqdm(dataloader):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    output = model(input_ids, attention_mask)
    utterances = [dataset.tokenizer.decode(input_id, skip_special_tokens=True) for input_id in input_ids]
    for utterance, label in zip(utterances, output.logits):
        id_str = model.model.config.id2label[label.argmax().item()]
        print("Text: {}\nEmotion: {}\n".format(utterance, id_str))
    break

  0%|          | 0/2714 [00:00<?, ?it/s]

Text: I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!
Emotion: remorse

Text: It's wonderful because it's awful. At not with.
Emotion: admiration



In [ ]:
all_baseline_scores = get_emotion_scores([
    "identity", "random", "word", "phrase", "sentence", "clustering", "archipelago"
])

SamLowe/roberta-base-go_emotions


100%|██████████| 1357/1357 [1:15:10<00:00,  3.32s/it]


In [ ]:
for name, score in all_baseline_scores.items():
    print(f'BASELINE {name} mean score: {score.mean()}')

BASELINE identity mean score: 0.010318499989807606
BASELINE random mean score: 0.029766259714961052
BASELINE word mean score: 0.11819196492433548
BASELINE phrase mean score: 0.019752763211727142
BASELINE sentence mean score: 0.011996912769973278
BASELINE clustering mean score: 0.08527955412864685
BASELINE archipelago mean score: 0.052712421864271164


In [ ]:
from lime.lime_text import LimeTextExplainer
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

In [ ]:
import random
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=list(model.model.config.id2label.values()))

def lime_predict(texts):
    """
    LIME expects a function that takes a list[str] and returns an array
    of shape (batch_size, num_classes) with predicted probabilities.
    """
    encoded = dataset.tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        logits = model(encoded['input_ids'], encoded['attention_mask']).logits
        probs = torch.softmax(logits, dim=-1)

    return probs.cpu().numpy()


In [ ]:
def lime_features_to_masks(lime_features, word_list):
    """
    lime_features: list of LIME features like [('angry', 0.35), ('so angry', 0.28)]
    word_list: list of tokens from your dataset

    returns: list of masks (list[list[int]])
    """
    masks = []

    for feat, weight in lime_features:
        feat_tokens = feat.split()                     # e.g. "so angry" -> ["so","angry"]
        mask = [0] * len(word_list)

        # slide over the sentence to match phrases
        for i in range(len(word_list) - len(feat_tokens) + 1):
            # match phrase sequentially
            if word_list[i:i+len(feat_tokens)] == feat_tokens:
                for j in range(len(feat_tokens)):
                    mask[i + j] = 1

        masks.append(mask)

    return masks


In [ ]:
lime_results = []

for idx in random.sample(range(len(dataset)), 10):
    item = dataset[idx]
    raw_text = dataset.dataset[idx]['text']
    word_list = item['word_list']

    exp = explainer.explain_instance(
        raw_text,
        classifier_fn=lime_predict,
        num_features=10,
        top_labels=1
    )

    pred_label = exp.top_labels[0]

    lime_feats = exp.as_list(label=pred_label)

    masks = lime_features_to_masks(lime_feats, word_list)

    lime_results.append({
        "word_list": word_list,
        "lime_features": lime_feats,
        "lime_masks": masks
    })


In [ ]:
print(lime_results)

In [ ]:
metric = EmotionFixScore()

lime_fix_scores = []
for entry in lime_results:
    masks = entry["lime_masks"]
    words = entry["word_list"]
    score = metric(masks, words)
    lime_fix_scores.append(score)

lime_fix_scores = torch.tensor(lime_fix_scores)


In [ ]:
all_scores = {
    "LIME": lime_fix_scores,
}

for name, score in all_scores.items():
    print(f'{name} mean score: {score.mean()}')


LIME mean score: 0.12307778745889664


In [ ]:
class TextCAM:
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.hidden = None

        # Hook embedding output
        emb = model.model.get_input_embeddings()
        emb.register_forward_hook(self.forward_hook)
        emb.register_backward_hook(self.backward_hook)

    def forward_hook(self, module, input, output):
        # output = (batch, seq_len, hidden)
        self.hidden = output

    def backward_hook(self, module, grad_input, grad_output):
        # grad_output[0] = gradient wrt embedding output
        self.gradients = grad_output[0]

    def compute_cam(self):
        """
        Returns a numpy CAM of shape (seq_len,)
        """
        # shape: (seq_len, hidden)
        H = self.hidden[0].detach()
        G = self.gradients[0].detach()

        # Grad-CAM for text = elementwise * then sum hidden dimension
        cam = (H * G).sum(dim=-1)

        # Normalize to 0–1
        cam = cam - cam.min()


In [ ]:
textcam = TextCAM(model)

for batch in tqdm(dataloader):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)

    # Forward pass
    outputs = model(input_ids, attention_mask)

    probs = torch.softmax(outputs.logits, dim=-1)
    pred_classes = probs.argmax(dim=-1)

    # Compute gradients wrt predicted class
    model.zero_grad()
    target = outputs.logits.gather(1, pred_classes.unsqueeze(1)).sum()
    target.backward()

    # Decode text
    utterances = [
        dataset.tokenizer.decode(ids, skip_special_tokens=True)
        for ids in input_ids
    ]

    for i, (utt, pred) in enumerate(zip(utterances, pred_classes)):
        cam = textcam.compute_cam()
        tokens = dataset.tokenizer.tokenize(utt)

        print("\nTEXT:", utt)
        print("PRED:", model.model.config.id2label[pred.item()])
        print("TEXT-CAM:")

        for tok, score in zip(tokens, cam[:len(tokens)]):
            print(f"{tok:15}  {score:.3f}")

    break


  0%|          | 0/2714 [00:00<?, ?it/s]


TEXT: I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!
PRED: remorse
TEXT-CAM:


TypeError: 'NoneType' object is not subscriptable

In [ ]:
def cam_to_mask(cam, words, threshold=0.4):
    mask = [1 if cam[i] > threshold else 0 for i in range(len(words))]
    return mask


In [ ]:
score = metric([mask], words)